# CHAMELEON layout student: inference only

Uses an adapter you already trained with `chameleon_layout_distill.ipynb`. No training happens here. It writes layouts for a prompts file.

1. Upload **`demo_prompts.jsonl`** (from `distil-research/training-data/llm-distill/`) to **`MyDrive/chameleon-distill/`**.
2. **Runtime → Change runtime type → T4 or L4** (inference only, so a T4 is fine).
3. **Runtime → Run all**. 56 layouts take about 10 minutes. Re-running skips finished ones.
4. Download **`demo_outputs_<model>.jsonl`** from the Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json
D = '/content/drive/MyDrive/chameleon-distill'
TAG = 'Qwen3-4B-Instruct-2507_v2'              # the trained run to use (its lora_adapter_<TAG> folder)
ADAPTER = f'{D}/lora_adapter_{TAG}'
PROMPTS = f'{D}/demo_prompts.jsonl'
OUT = f'{D}/demo_outputs_{TAG}.jsonl'
assert os.path.isdir(ADAPTER), f'missing {ADAPTER}'
assert os.path.exists(PROMPTS), 'upload demo_prompts.jsonl to MyDrive/chameleon-distill first'

In [ ]:
!pip -q install unsloth

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(model)

In [ ]:
SYSTEM = 'You are an ad layout engine. Given a source ad as JSON and a target canvas, return the resized layout as JSON.'
done = {json.loads(l)['cell'] for l in open(OUT)} if os.path.exists(OUT) else set()
with open(OUT, 'a') as f:
    for r in map(json.loads, open(PROMPTS)):
        if r['cell'] in done: continue
        msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':r['prompt']}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
        gen = model.generate(input_ids=ids, max_new_tokens=1500, do_sample=False)
        text = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
        f.write(json.dumps({'cell': r['cell'], 'output': text}) + '\n'); f.flush()
        print(r['cell'], len(text), 'chars')
print('done ->', OUT)